# Calibrated Closed-Loop Posterior-BDDM CUDA Run

Runtime: choose **GPU** before running. This notebook runs the calibrated closed-loop hierarchy using the oracle-calibration settings identified from the previous sweep: `h=0.05`, `beta=0`, `n_steps=80`.

In [ ]:
REPO_URL = "https://github.com/Seif-Hussein/blind-diffusion-toy.git"
BRANCH = "agent/colab-cuda-oracle"  # Change to "master" after merge.

!rm -rf blind-diffusion-toy
!git clone --depth 1 --branch {BRANCH} {REPO_URL} blind-diffusion-toy
%cd blind-diffusion-toy
!python -m pip install -q -r posterior_bddm_oracle/requirements-colab.txt

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))

## Run Calibrated Closed-Loop Hierarchy

This compares the posterior oracle, exact-correction oracle, blind split, scheduled split, scheduled finite-dual PDHG split, posterior-scale split, naive blind force, and raw HQS under calibrated oracle settings. The eta grid is narrowed around the useful range from the previous run.

In [ ]:
!python -m posterior_bddm_oracle.src.experiments_posterior_closed_loop_torch \
  --device auto \
  --prior ellipse \
  --d-values 100,500 \
  --eta-values 0.05,0.075,0.1,0.15,0.2 \
  --measurement-ratios 0.5 \
  --noise-stds 0.08 \
  --n-trials 128 \
  --n-steps 80 \
  --init posterior \
  --h 0.05 \
  --beta 0 \
  --split gradient \
  --pdhg-gamma 100 \
  --out posterior_bddm_oracle/results_cuda_closed_loop_calibrated_baseline

In [ ]:
from pathlib import Path
report = Path("posterior_bddm_oracle/results_cuda_closed_loop_calibrated_baseline/data/closed_loop_cuda_report.md")
print(report.read_text() if report.exists() else "Calibrated closed-loop report not found")

## Zip Results

In [ ]:
from pathlib import Path
import zipfile

target = Path("posterior_bddm_oracle/results_cuda_closed_loop_calibrated_baseline")
zip_path = Path("posterior_bddm_closed_loop_calibrated_baseline.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in target.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=path)

print(f"Created {zip_path.resolve()}")
print(f"Size: {zip_path.stat().st_size / (1024**2):.2f} MB")

In [ ]:
from google.colab import files
files.download("posterior_bddm_closed_loop_calibrated_baseline.zip")